# Logistic Function (Sigmoid Function)

The **logistic function**, also known as the **sigmoid function**, transforms a linear combination of inputs into a value between **0** and **1**. It is commonly used in **logistic regression** to model probabilities.

The logistic function is defined as:

$$
y = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x)}}
$$

where:

- $y$ is the predicted probability,
- $x$ is the input variable,
- $\beta_0$ is the intercept,
- $\beta_1$ is the coefficient (weight),
- $e$ is Euler's number ($e \approx 2.71828$).

### Interpretation

The term

$$
\beta_0 + \beta_1 x
$$

can take any value between $-\infty$ and $+\infty$. The logistic function converts this value into a probability:

- If $\beta_0 + \beta_1 x$ is very large, then $y \approx 1$.
- If $\beta_0 + \beta_1 x$ is very small, then $y \approx 0$.
- If $\beta_0 + \beta_1 x = 0$, then $y = 0.5$.

Therefore, the logistic function is ideal for binary classification problems, where the goal is to predict the probability of one of two possible outcomes.

In [5]:
from math import exp
def sigmoid(x,b0,b1):
    return 1 / ( 1 + exp(-(b0+b1*x)))

sigmoid(0,5,3)

0.9933071490757153

In [7]:
from sympy import symbols, exp

b0, b1, x = symbols("b0 b1 x")

sigmoid = 1 / (1.0 + exp(-(b0 + b1 * x)))

print(sigmoid.subs(b0,-2.931).subs(b1,.06))

1/(1.0 + 18.7463674941069*exp(-0.06*x))


In [12]:
# simple logistic regression

import pandas as pd
from sklearn.linear_model import LogisticRegression
# Load the data
df = pd.read_csv('https://raw.githubusercontent.com/thomasnield/machine-learning-demo-data/master/classification/simple_logistic_regression.csv', delimiter=",")
# Extract input variables (all rows, all columns but last column)
X = df.values[:, :-1]
# Extract output column (all rows, last column)
Y = df.values[:, -1]

print(X,Y)
# Perform logistic regression
# Turn off penalty
model = LogisticRegression()
model.fit(X, Y)
# print beta1
print(model.coef_.flatten()) # 0.69267212
# print beta0
print(model.intercept_.flatten()) # -3.17576395

[[1. ]
 [1.5]
 [2.1]
 [2.4]
 [2.5]
 [3.1]
 [4.2]
 [4.4]
 [4.6]
 [4.9]
 [5.2]
 [5.6]
 [6.1]
 [6.4]
 [6.6]
 [7. ]
 [7.6]
 [7.8]
 [8.4]
 [8.8]
 [9.2]] [0. 0. 0. 0. 1. 0. 0. 1. 1. 0. 1. 0. 1. 1. 1. 0. 1. 1. 1. 1. 1.]
[0.63943018]
[-2.91592947]


## Joint Likelihood in Logistic Regression

The logistic regression model gives:

$$
p_i = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_i)}}
$$

where:

$$
p_i = P(y_i = 1 \mid x_i)
$$

So:

$$
1 - p_i = P(y_i = 0 \mid x_i)
$$

For one observation, the likelihood is:

$$
p_i^{y_i}(1-p_i)^{1-y_i}
$$

This formula means:

If the real value is:

$$
y_i = 1
$$

then:

$$
p_i^{1}(1-p_i)^0 = p_i
$$

If the real value is:

$$
y_i = 0
$$

then:

$$
p_i^{0}(1-p_i)^1 = 1-p_i
$$

So the formula automatically keeps the probability of the label that actually happened.

For all observations, we multiply these probabilities:

$$
L(\beta_0,\beta_1)
=
\prod_{i=1}^{n}
p_i^{y_i}(1-p_i)^{1-y_i}
$$

This answers the question:

**How probable is it that the model would produce the labels we actually observed?**

A higher likelihood means the model gives high probabilities to the correct observed labels.

In [9]:
import pandas as pd
from math import exp
df = list(pd.read_csv("https://raw.githubusercontent.com/thomasnield/machine-learning-demo-data/master/classification/simple_logistic_regression.csv").itertuples())

b0 = -3.17576395
b1 = 0.69267212

def  sigmoid(x,b0,b1):
    return 1.0 / ( 1 + exp(-(b0 + b1 * x)))
likelihood = 1
for point in df:
    if point.y == 1:
        likelihood  = likelihood * sigmoid(point.x,b0,b1)
    else:
        likelihood  = likelihood *  (1 -sigmoid(point.x,b0,b1))
print(likelihood)
        

4.7911180221699105e-05


In [26]:
import pandas as pd
from sympy import Sum,symbols,diff,Function, exp, log,lambdify

points = list(pd.read_csv("https://raw.githubusercontent.com/thomasnield/machine-learning-demo-data/master/classification/simple_logistic_regression.csv").itertuples())

b1, b0, i, n = symbols('b1 b0 i n')
x, y = symbols('x y', cls=Function)
joint_likelihood = Sum(log((1.0 / (1.0 + exp(-(b0 + b1 * x(i))))) ** y(i) \
* (1.0 - (1.0 / (1.0 + exp(-(b0 + b1 * x(i)))))) ** (1 - y(i))), (i, 0,n))
# Partial derivative for m, with points substituted
d_b1 = diff(joint_likelihood, b1) \
 .subs(n, len(points) - 1).doit() \
 .replace(x, lambda i: points[i].x) \
 .replace(y, lambda i: points[i].y)

# Partial derivative for m, with points substituted
d_b0 = diff(joint_likelihood, b0) \
 .subs(n, len(points) - 1).doit() \
 .replace(x, lambda i: points[i].x) \
 .replace(y, lambda i: points[i].y)

# compile using lambdify for faster computation
d_b1 = lambdify([b1, b0], d_b1)
d_b0 = lambdify([b1, b0], d_b0)

# Perform Gradient Descent
b1 = 0.01
b0 = 0.01
L = .01

for j in range(10_000):
    b1 += d_b1(b1, b0) * L
    b0 += d_b0(b1, b0) * L
print(b1, b0)
# 0.6926693075370812 -3.175751550409821

0.6926693075370812 -3.175751550409821


In [ ]:
# multi variable logistic regression
import pandas as pd
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("https://raw.githubusercontent.com/thomasnield/machine-learning-demo-data/master/classification/employee_retention_analysis.csv")

X = df.values[:,:-1]
y = df.values[:,-1]

model = LogisticRegression()

model.fit(X,y)

# Print coefficients:
print("COEFFICIENTS: {0}".format(model.coef_.flatten()))
print("INTERCEPT: {0}".format(model.intercept_.flatten()))

# Interact and test with new employee data
def predict_employee_will_stay(sex, age, promotions, years_employed):
    prediction = model.predict([[sex, age, promotions, years_employed]])
    probabilities = model.predict_proba([[sex, age, promotions, years_employed]])
    if prediction == [[1]]:
        return "WILL LEAVE: {0}".format(probabilities)
    else:
        return "WILL STAY: {0}".format(probabilities)
# Test a prediction
while True:
 n = input("Predict employee will stay or leave {sex},{age},{promotions},{years employed}: ")
 (sex, age, promotions, years_employed) = n.split(",")
 print(predict_employee_will_stay(int(sex), int(age), int(promotions),int(years_employed)))


COEFFICIENTS: [-1.19726176e-03  2.80082089e-02 -1.72134783e+00  6.63608882e-01]
INTERCEPT: [-1.96223966]


Predict employee will stay or leave {sex},{age},{promotions},{years employed}:  1,26,0,1


WILL STAY: [[0.63913565 0.36086435]]


Predict employee will stay or leave {sex},{age},{promotions},{years employed}:  1,25,0,2


WILL LEAVE: [[0.48400894 0.51599106]]


# Pseudo R² in Logistic Regression

Unlike Linear Regression, Logistic Regression does not have a true R² because the target variable is binary (0 or 1).

Instead, we use **Pseudo R²**, which measures how much better the model performs compared to a null model.

## McFadden's R²

$$
R^2 = 1 - \frac{\log L_{\text{model}}}{\log L_{\text{null}}}
$$

where:

- $\log L_{\text{model}}$ = log-likelihood of the fitted model
- $\log L_{\text{null}}$ = log-likelihood of the null model (no features)

## Intuition

The null model predicts only the overall probability of class 1.

Pseudo R² measures:

> How much better is the logistic regression model than the null model?

- Higher values indicate a better model.
- A value of 0 means the model is no better than the null model.
- Values between 0.2 and 0.4 are often considered good for logistic regression.

In [8]:
import pandas as pd
from math import log, exp
points = list(pd.read_csv("https://raw.githubusercontent.com/thomasnield/machine-learning-demo-data/master/classification/simple_logistic_regression.csv").itertuples())

p_of_1 = sum(p.y for p in points) / len(points)
l_null = sum(log(p_of_1) * p.y + log(1 - p_of_1) *(1 - p.y) for p in points)

b0 = -3.17576395
b1 = 0.69267212

def sigmoid(x,b0,b1):
    return 1.0 / ( 1.0 + exp(-(b0 + b1 *x)))
    
l_model = sum((log(sigmoid(p.x,b0,b1)) * p.y + log(1 - sigmoid(p.x,b0,b1)) * (1 - p.y)) for p in points)

r2 = (l_null - l_model) / l_null

print(r_squared)

-0.6600022310832683
